# Fuzzy Asteroids

In this notebook you'll build an AI pilot for the Kessler asteroids game out of **fuzzy logic**: a handful of rules written almost the way a person would say them.

- *If the nearest asteroid is to my left, turn left.*
- *If it's straight ahead, fire.*
- *If it's close, back away.*

A plain `if` statement would snap between "turn" and "don't turn", making the ship jerk around. Fuzzy rules blend into each other, so the pilot turns hard when it's far off target and eases off as it lines up, just as a human pilot would. This is the same idea teams use in the Explainable Fuzzy Competition, where Kessler pilots have to be good *and* able to explain their decisions.

This notebook combines two libraries. If either is new to you, start with [[A156BED9-159E-47BB-8B36-FE8125CAE57E|Scikit-fuzzy]] or [[47464124-DC6E-4543-ABBA-B2223F4A1B44|Kessler Game]].

# What the Pilot Senses

Each frame, the pilot looks at the **nearest asteroid** and measures two things. These are the fuzzy inputs, or **antecedents**:

- **aim**: how far the ship must turn to point at it, from -180 to 180 degrees. Negative means the asteroid is to the right, positive to the left, and near 0 straight ahead.
- **distance**: how far away it is. Anything beyond 600 counts as 600.

Each input gets three overlapping groups. "Ahead" is a narrow triangle, because the ship has to be lined up accurately before a bullet will hit. The first cell plots the aim groups, the second the distance groups.

In [ ]:
import math
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

aim = ctrl.Antecedent(np.linspace(-180, 180, 73), 'aim')
distance = ctrl.Antecedent(np.linspace(0, 600, 61), 'distance')

aim['right'] = fuzz.trapmf(aim.universe, [-180, -180, -30, 0])
aim['ahead'] = fuzz.trimf(aim.universe, [-15, 0, 15])
aim['left'] = fuzz.trapmf(aim.universe, [0, 30, 180, 180])

distance['close'] = fuzz.trapmf(distance.universe, [0, 0, 80, 160])
distance['medium'] = fuzz.trimf(distance.universe, [80, 200, 320])
distance['far'] = fuzz.trapmf(distance.universe, [240, 360, 600, 600])

aim.view()

In [ ]:
distance.view()

# What the Pilot Does

The outputs, or **consequents**, are the controls the game asks for:

- **turn**: degrees per second, positive for left (anticlockwise).
- **thrust**: forwards is positive. We only use half the ship's full power, so it stays easy to control.
- **fire**: a score from 0 to 1. The pilot shoots whenever it comes out above 0.5.

Making "fire" fuzzy too means the decision to shoot comes from the same rules as the steering, rather than from a separate `if`.

In [ ]:
turn = ctrl.Consequent(np.linspace(-180, 180, 73), 'turn')
thrust = ctrl.Consequent(np.linspace(-240, 240, 49), 'thrust')
fire = ctrl.Consequent(np.linspace(0, 1, 11), 'fire')

turn['right'] = fuzz.trimf(turn.universe, [-180, -180, 0])
turn['none'] = fuzz.trimf(turn.universe, [-30, 0, 30])
turn['left'] = fuzz.trimf(turn.universe, [0, 180, 180])

thrust['back'] = fuzz.trimf(thrust.universe, [-240, -240, 0])
thrust['stop'] = fuzz.trimf(thrust.universe, [-60, 0, 60])
thrust['forward'] = fuzz.trimf(thrust.universe, [0, 240, 240])

fire['no'] = fuzz.trimf(fire.universe, [0, 0, 1])
fire['yes'] = fuzz.trimf(fire.universe, [0, 1, 1])

turn.view()

# The Rules

Seven rules make the whole pilot. The first three steer and decide when to shoot. The rest handle speed: back away from close asteroids, hold still at medium range, and only fly towards distant ones when they're straight ahead.

A rule can set several outputs at once by giving a tuple of consequents, and `~` means NOT, so `~aim['ahead']` reads "not ahead". Because the groups overlap, several rules usually fire at once, and their outputs blend.

The `ControlSystemSimulation` is built once here and reused for every frame, because building it is far slower than running it.

In [ ]:
rules = [
  ctrl.Rule(aim['left'], (turn['left'], fire['no'])),
  ctrl.Rule(aim['right'], (turn['right'], fire['no'])),
  ctrl.Rule(aim['ahead'], (turn['none'], fire['yes'])),
  ctrl.Rule(distance['close'], thrust['back']),
  ctrl.Rule(distance['medium'], thrust['stop']),
  ctrl.Rule(distance['far'] & aim['ahead'], thrust['forward']),
  ctrl.Rule(distance['far'] & ~aim['ahead'], thrust['stop']),
]
pilot = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))
print(len(rules), "rules ready")

# Trying the Rules

Before letting the pilot fly, you can ask it what it *would* do. Drag the sliders to place an imaginary asteroid, and watch the three outputs change.

Start with an asteroid just to the left (AIM 5): the pilot turns gently and fires. Move it further left (AIM 40) and the turn grows smoothly while the fire score drops, because the asteroid has left the "ahead" triangle. Then slide DISTANCE down and watch the thrust go into reverse.

In [ ]:
AIM = 5 #@param {type:"slider", min:-180, max:180, step:5}
DISTANCE = 300 #@param {type:"slider", min:0, max:600, step:10}

pilot.input['aim'] = AIM
pilot.input['distance'] = DISTANCE
pilot.compute()

print(f"turn:   {pilot.output['turn']:6.1f} degrees per second")
print(f"thrust: {pilot.output['thrust']:6.1f}")
print(f"fire:   {pilot.output['fire']:6.2f}  ->  {'FIRE' if pilot.output['fire'] > 0.5 else 'hold'}")
turn.view(sim=pilot)

# The Fuzzy Pilot

Now the rules become a Kessler controller. Each frame, `actions()` finds the nearest asteroid, measures the aim and distance, runs the fuzzy system and hands its outputs to the game.

The game insists that `fire` is a real Python `True` or `False`. Comparing a fuzzy output with `> 0.5` gives a NumPy value instead, so it's wrapped in `bool(...)`.

In [ ]:
from kesslergame import KesslerGame, TrainerEnvironment, Scenario, KesslerController
from kesslergame.state_models import ShipState, GameState

class FuzzyPilot(KesslerController):
  name = "Fuzzy Pilot"

  def actions(self, ship_state: ShipState, game_state: GameState):
    x, y = ship_state.position
    nearest = min(game_state.asteroids, key=lambda a: math.dist(a.position, (x, y)))
    target = math.degrees(math.atan2(nearest.y - y, nearest.x - x))

    pilot.input['aim'] = (target - ship_state.heading + 180) % 360 - 180
    pilot.input['distance'] = min(math.dist(nearest.position, (x, y)), 600)
    pilot.compute()

    out = pilot.output
    return float(out['thrust']), float(out['turn']), bool(out['fire'] > 0.5), False

# Putting It to the Test

How good is it? Let's race it against a simpler, non-fuzzy controller, the **Aimer**, which turns towards the nearest asteroid and fires when it's within 10 degrees, but never moves.

Both pilots play the same five 30-second matches without graphics (each `seed` gives the same asteroid layout to both). Hits are good. Deaths are bad: three deaths ends the match early. The fuzzy pilot runs its rules about 4,500 times here, so this takes around ten seconds.

Notice the trade-off. The fuzzy pilot dies less often because it backs away from danger, but it hits slightly fewer asteroids.

In [ ]:
class Aimer(KesslerController):
  name = "Aimer"

  def actions(self, ship_state: ShipState, game_state: GameState):
    x, y = ship_state.position
    nearest = min(game_state.asteroids, key=lambda a: math.dist(a.position, (x, y)))
    target = math.degrees(math.atan2(nearest.y - y, nearest.x - x))
    error = (target - ship_state.heading + 180) % 360 - 180
    return 0.0, max(-180, min(180, error * 6)), abs(error) < 10, False

def match(seed):
  return Scenario(num_asteroids=8, ship_states=[{"position": (500, 400), "lives": 3}], time_limit=30, seed=seed)

for controller in [Aimer(), FuzzyPilot()]:
  hits = deaths = 0
  for seed in range(5):
    score, perf = TrainerEnvironment().run(scenario=match(seed), controllers=[controller])
    hits += score.teams[0].asteroids_hit
    deaths += score.teams[0].deaths
  print(f"{controller.name:12} {hits / 5:5.1f} hits, {deaths / 5:.1f} deaths per match")

# Watch It Play

Time to see the pilot fly. This runs a 20-second match on screen. Watch how it backs away when an asteroid drifts close, and how it eases into each shot instead of overshooting.

When the match ends, the console tab (>_) under the game shows how it did.

In [ ]:
show = Scenario(num_asteroids=8, ship_states=[{"position": (500, 400), "lives": 3}], time_limit=20, seed=1)
score, perf = KesslerGame().run(scenario=show, controllers=[FuzzyPilot()])

team = score.teams[0]
print(f"{team.asteroids_hit} asteroids hit, {team.accuracy:.0%} accuracy, {team.deaths} deaths")

# Things to Try

- **Get the best of both.** Can you change the rules so the fuzzy pilot keeps its low death count *and* beats the Aimer's hits? Re-run the test cell to check.
- **Change a group.** Widen `aim['ahead']` to `[-30, 0, 30]` and re-run from the inputs cell down. Does the pilot shoot more or hit less?
- **Add a rule.** The pilot ignores asteroids flying *towards* it. Add an input for how fast the nearest asteroid is approaching (use its `velocity`), and a rule to back away from fast ones.
- **Lead the target.** Aim at where the asteroid *will be* when the bullet arrives, not where it is now.
- **Use mines.** Give the ship `"mines_remaining": 3` in the scenario, add a `mine` consequent, and return it as the fourth value of `actions()`.
- **Race it yourself.** Add a second ship with `"team": 2` and pass `[FuzzyPilot(), KeyboardController()]` (from `kesslergame`) to see whether you can beat it.

# Check Your Understanding